In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hardikmanglani/rhyme-corpus-1000-en/rhyme_corpus_1000_en.txt
/kaggle/input/datasets/hardikmanglani/corpuse/rhyme_corpus_en.txt
/kaggle/input/datasets/hardikmanglani/new-dataset/new_rhyme_ground_truth.csv


In [2]:
import os
import re
import nltk

# Download the CMU Pronouncing Dictionary resource from NLTK
nltk.download('cmudict')
from nltk.corpus import cmudict

# Set up your dataset location path 
# (Update this path to match your actual Kaggle dataset folder name)
CORPUS_FILE = "/kaggle/input/datasets/hardikmanglani/new-dataset/new_rhyme_ground_truth.csv"

# Initialize the offline dictionary lookup database
d = cmudict.dict()

# Quick configuration check
if os.path.exists(CORPUS_FILE):
    print("✅ Success: Found corpus file path!")
else:
    print("❌ Error: Corpus file not found. Check your Kaggle data input paths.")

[nltk_data] Downloading package cmudict to /usr/share/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


✅ Success: Found corpus file path!


In [3]:
def extract_cmu_rhyme_part(phonemes):
    """
    Finds the primary stressed vowel (marked with '1') in the ARPAbet phone list
    and slices everything from that vowel sound to the end of the word.
    """
    for index, phone in enumerate(phonemes):
        # Check if the phoneme contains a '1' indicating primary stress (e.g., 'AY1', 'EH1')
        if '1' in phone:
            return phonemes[index:]
    
    # Fallback: if no primary stress is flagged, look for any syllable stress variants ('2' or '0')
    for index, phone in enumerate(phonemes):
        if any(char.isdigit() for char in phone):
            return phonemes[index:]
            
    # Ultimate fallback: return the last 2 phonemes if no stress marks are found
    return phonemes[-2:] if len(phonemes) >= 2 else phonemes

def deterministic_rhyme_nltk(word1, word2):
    """Determines if two words rhyme using NLTK CMUDict completely offline."""
    w1 = word1.lower().strip()
    w2 = word2.lower().strip()
    
    if w1 == w2:
        return False
        
    # Get phonetic variations from dictionary entries
    pronunciations1 = d.get(w1)
    pronunciations2 = d.get(w2)
    
    # If any of the words are completely missing from CMUDict, we cannot evaluate
    if not pronunciations1 or not pronunciations2:
        return False
        
    # Check all possible pronunciations variations (e.g., words with multiple valid accents)
    for p1 in pronunciations1:
        for p2 in pronunciations2:
            rhyme_part1 = extract_cmu_rhyme_part(p1)
            rhyme_part2 = extract_cmu_rhyme_part(p2)
            
            # If the calculated rhyming extensions match perfectly
            if rhyme_part1 == rhyme_part2:
                return True
                
    return False

In [4]:
total_pairs = 0
correct_predictions = 0
skipped_lines = 0

# Metric counters
true_positives = 0
false_positives = 0
true_negatives = 0
false_negatives = 0

print("Evaluating corpus entries locally via NLTK...")

with open(CORPUS_FILE, "r", encoding="utf-8") as file:
    for line in file:
        line_str = line.strip()
        if not line_str:
            continue
            
        # Clean out metadata artifacts if present
        line_str = re.sub(r'\\', '', line_str).strip()
        
        parts = line_str.split(',')
        if len(parts) != 3:
            skipped_lines += 1
            continue
            
        word1 = parts[0].strip()
        word2 = parts[1].strip()
        
        try:
            ground_truth = int(parts[2].strip())  # 1 = Rhyme, 0 = Non-Rhyme
        except ValueError:
            skipped_lines += 1
            continue
            
        # Compute algorithm output
        is_rhyme = deterministic_rhyme_nltk(word1, word2)
        predicted_truth = 1 if is_rhyme else 0
        
        # Calculate scores
        total_pairs += 1
        if predicted_truth == ground_truth:
            correct_predictions += 1
            
        if ground_truth == 1 and predicted_truth == 1:
            true_positives += 1
        elif ground_truth == 0 and predicted_truth == 1:
            false_positives += 1
        elif ground_truth == 0 and predicted_truth == 0:
            true_negatives += 1
        elif ground_truth == 1 and predicted_truth == 0:
            false_negatives += 1

# Display results
if total_pairs > 0:
    accuracy = (correct_predictions / total_pairs) * 100
    print("\n================ NLTK OFFLINE EVALUATION ================")
    print(f"Total Evaluated Pairs : {total_pairs}")
    print(f"Correct Predictions   : {correct_predictions}")
    print(f"Skipped Lines         : {skipped_lines}")
    print(f"Final Model Accuracy  : {accuracy:.2f}%")
    print("---------------------------------------------------------")
    print(f"True Rhymes Caught (TP)       : {true_positives}")
    print(f"False Rhymes Mismatched (FP)  : {false_positives}")
    print(f"True Non-Rhymes Scored (TN)   : {true_negatives}")
    print(f"Missed Rhymes (FN)            : {false_negatives}")
    print("=========================================================")
else:
    print("Failed to run: No usable data found in your corpus configuration path.")

Evaluating corpus entries locally via NLTK...

================ NLTK OFFLINE EVALUATION ================
Total Evaluated Pairs : 8168623
Correct Predictions   : 8166543
Skipped Lines         : 1
Final Model Accuracy  : 99.97%
---------------------------------------------------------
True Rhymes Caught (TP)       : 33720
False Rhymes Mismatched (FP)  : 2080
True Non-Rhymes Scored (TN)   : 8132823
Missed Rhymes (FN)            : 0
